# Suite CTY — The C Types dialect

`Expressions.C.Types` models C's type names (ISO/IEC 9899, 6.7.7 in C11, "Type names"), a grammar of their own, as
mbse-programs has them: a type name's specifiers and its abstract declarator, and an enumeration's body, its enumerators
and the directives and comments among them, as data. C's terms hold its terms where a type stands (a cast's, a compound
literal's, a generic association's type) and where an expression or a type may (`sizeof(int)`), C reading them. Its
kinds are validated, bind nothing, cover C's three editions, print as mbse-programs prints them, and are refused by
evaluation by name; a cast reads its type from it.

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.C import Expressions as CFamily, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.Cpp import Expressions as CppFamily, Types as CppTypes
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.C.Expressions import Simplifiers as CV, Expressions as C, Text as CText
from mbse.Expressions.C.Preprocessor import Expressions as PP
from mbse.Expressions.C.Types import Expressions as CT, Text as CTText
from mbse.Expressions.Cpp.Types import Expressions as XT
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = Builders.Store()  # the C family's dialects, C Types' install before C's: the installs run in any order
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, CTypes.install, CFamily.install, CppFamily.install,
                CppTypes.install, CPreprocessor.install, AMD64.install]:
    store.transaction(install)
for define in [CFamily.define, CppFamily.define, AMD64.define, CTypes.define]:  # C's types and the ABI's, C Types' editions
    store.transaction(define)
AMD64_ = store.Instances.Expressions.SystemV.AMD64  # the ABI a holder names, whose types C leaves to the implementation


def term(spec, dialect=CT.DIALECT, abi=AMD64_):
    """The term value `spec` writes, the root of `dialect`'s holder, naming `abi`, written in a transaction."""
    with store.transaction():
        path = dialect.path.split(".")
        holder = getattr(getattr(store.Expressions, path[0]), path[1]).Expression
        return F.term(holder(lambda e: e.root(dialect.spec(spec)).abi([abi]) if abi is not None else e.root(dialect.spec(spec))))


def validate(spec, bound=(), dialect=CT.DIALECT):
    return dialect.validate(term(spec, dialect), bound)


text = lambda spec, dialect=CT.DIALECT: CTText.ToText(term(spec, dialect))  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
schemas = lambda name: {b.name: b.schema for b in store.registered(name).branches}  # noqa: E731
spelled = lambda word: lambda i: i.spelling(word)  # noqa: E731: an identifier, of its spelling
named = lambda word: lambda n: n.identifier(spelled(word))  # noqa: E731: a name that is an identifier
keyword = lambda word: lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))  # noqa: E731
type_name = lambda *specifiers, declarator=None: lambda t: t.typeid(  # noqa: E731: a type name, of its specifiers
    lambda i: i.specifiers(list(specifiers)) if declarator is None else i.specifiers(list(specifiers)).declarator(declarator))
int_ = type_name(keyword("int"))
x = C.identifier("x")
enumerator = lambda word, value=None: lambda u: u.enumerator(  # noqa: E731: `word = value`
    lambda r: r.name(spelled(word)) if value is None else r.name(spelled(word)).value(value))
enumeration = lambda items, trailing=False, word="E": type_name(lambda s: s.enumspecifier(  # noqa: E731: `enum E { items }`
    lambda e: e.key("enum").name(named(word)).body(True).enumerators(items).trailing_comma(trailing)))
ifdef = lambda word, items: CT.DIALECT.spec(lambda u: u.ifdefdirective(lambda d: d.name(spelled(word)).items(  # noqa: E731
    [CT.DIALECT.spec(item) for item in items])))  # `#ifdef word`, its items, C Types' terms, where an enumeration holds it


<a id="CTY-01"></a>
## CTY-01 · Its union and holder: its kinds, a type name and an enumerator, invalid syntax, and the preprocessor's directives and comment, an enumeration's items; a holder refining the framework's, naming its ABI, round-tripped through JSON

In [ ]:
# Its Term: its kinds, mbse-programs', then invalid syntax, then C Preprocessor's directives and comment, which its
# install adds, an enumeration's items; C's Term lists its typeid, for a subterm that may be an expression or a type
assert CT.TAGS == ("typeid", "enumerator") and list(CT.DIALECT.kinds()) == [*CT.TAGS, "error", "missing"]
assert union(CT.DIALECT.union) == [*CT.TAGS, "error", "missing", *PP.DIRECTIVES, "comment"]
assert CT.DIALECT.name() == "C Types" and CT.DIALECT.tag == "c_types" and "c_types" in union("Expressions.Framework.Term")
kinds, held = schemas(CT.DIALECT.union), schemas(C.DIALECT.union)
assert held["typeid"] is kinds["typeid"] is store.registered("Expressions.C.Types.TypeId")  # one schema, in each union
assert "enumerator" not in held and all(kinds[tag] is schemas(PP.DIALECT.union)[tag] for tag in PP.DIRECTIVES)
assert CT.DIALECT.reads() == (C.DIALECT, PP.DIALECT) and C.DIALECT.kind_named("typeid") is CT.DIALECT.kinds()["typeid"]
# Its holder: a reference object refining the framework's Expression, its root narrowed to its Term, naming its ABI.
Holder = store.registered("Expressions.C.Types.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(CT.DIALECT.union) and list(Holder.adjacencies) == ["abi"]
color = enumeration([enumerator("RED", C.literal(1)), enumerator("GREEN")])
with store.transaction():
    holder = store.Expressions.C.Types.Expression(lambda e: e.root(CT.DIALECT.spec(color)).abi([AMD64_]))
    linked = store.Schemas.OfExpression(lambda s: s.name("cty.color").expression(lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and CT.DIALECT.validate(holder) == []
assert F.dialect_of(F.term(holder)) == CT.DIALECT.union and F.kind_of(F.term(holder))[0] == "typeid"
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
assert [entry.abi for entry in holder.abi] == [AMD64_]
with raises(Exception, match="'binaryexpression' is not a member of a union value"):  # its root holds its own terms
    with store.transaction():
        store.Expressions.C.Types.Expression(lambda e: e.root(CT.DIALECT.spec(C.binary("+", x, 1))))


<a id="CTY-02"></a>
## CTY-02 · Its kinds and helpers: mbse-programs' properties in its order, a type name's parts its helpers, an enumeration's body its terms, the terms they hold C's, and the subterms of C that are its terms

In [ ]:
# Its kinds' properties, as mbse-programs has them, and an enumeration's body: whether it has one, its items, its
# trailing comma (mbse-programs' `EnumeratorList`, omitted as a wrapper)
props = lambda name: list(store.registered(f"Expressions.C.Types.{name}").properties)  # noqa: E731
assert props("TypeId") == ["specifiers", "declarator"] and props("Enumerator") == ["name", "attributes", "value"]
assert props("EnumSpecifier") == ["key", "attributes", "name", "base", "body", "enumerators", "trailing_comma"]
assert props("ArrayDeclarator") == ["declarator", "static", "qualifiers", "size", "star", "attributes"]
# Its helpers, a type name's parts, which no expression of C holds, each union named after mbse-programs' category
assert CT.HELPERS == ("DeclSpecifier", "CvQualifier", "PrimitiveTypeSpecifier", "NamedTypeSpecifier", "ClassSpecifier",
                      "EnumSpecifier", "PlaceholderTypeSpecifier", "TypeofSpecifier", "AtomicTypeSpecifier", "BitIntSpecifier",
                      "IdDeclarator", "PointerDeclarator", "ArrayDeclarator", "FunctionDeclarator", "ParenthesizedDeclarator",
                      "ParameterDeclaration", "EllipsisParameter", "StandardAttributeSpecifier", "Attribute", "AlignasSpecifier",
                      "GnuAttributeSpecifier", "DeclspecSpecifier")
assert all(not store.registered(f"Expressions.C.Types.{helper}").ref and helper.lower() not in kinds for helper in CT.HELPERS)
assert union("Expressions.C.Types.Specifier")[-4:] == union("Expressions.C.Types.AttributeSpecifier")  # its specifiers hold them
assert union("Expressions.C.Types.Declarator") == ["iddeclarator", "pointerdeclarator", "arraydeclarator", "functiondeclarator",
                                                   "parenthesizeddeclarator"]
assert union("Expressions.C.Types.Parameter") == ["parameterdeclaration", "ellipsisparameter"]
# The terms a type name holds are C's, and its names C's Name: read through `reads`
CTerm, Name = store.registered(C.DIALECT.union), store.registered("Expressions.C.Expressions.Name")
prop = lambda schema, name: store.registered(f"Expressions.C.Types.{schema}").properties[name].schema  # noqa: E731
assert prop("ArrayDeclarator", "size") is CTerm and prop("TypeofSpecifier", "operand") is CTerm and prop("Enumerator", "value") is CTerm
assert prop("Attribute", "arguments").element is CTerm and prop("NamedTypeSpecifier", "name") is Name and prop("IdDeclarator", "name") is Name
assert prop("Enumerator", "name") is store.registered("Expressions.C.Expressions.Identifier")
assert prop("EnumSpecifier", "enumerators").element is store.registered(CT.DIALECT.union)  # its items, this dialect's terms
assert prop("EnumSpecifier", "base") is prop("AtomicTypeSpecifier", "type") is store.registered(CT.DIALECT.union)
# C's type subterms are of its Term: a cast's, a compound literal's and a generic association's, each a term
Types = store.registered(CT.DIALECT.union)
assert all(store.registered(f"Expressions.C.Expressions.{s}").properties["type"].schema is Types for s in (
    "CastExpression", "CompoundLiteralExpression", "GenericAssociation"))
with raises(LookupError, match="no schema registered as 'Expressions.C.Expressions.Type'"):  # moved, no alias
    store.registered("Expressions.C.Expressions.Type")
# A type name and an enumerator, as forms: a helper a Part, the terms its parts hold its arguments
assert F.form(term(int_)) == F.Form("typeid", {"specifiers": (F.Part("primitivetypespecifier", {"keyword": "int"}, 0),)})
assert F.form(term(enumerator("A", C.literal(1)))).attributes == {"name": F.Part("identifier", {"spelling": "A"}, 0)}
made = F.form(term(enumeration([enumerator("A", C.literal(1))])))
assert made.kind == "typeid" and made.arguments[0] is None and [F.kind_of(a)[0] for a in made.arguments[1:]] == ["enumerator"]
# its arguments: its specifier's, an enumeration's fixed type, empty, then its body's items


<a id="CTY-03"></a>
## CTY-03 · Its checks: an enumerator needs a name, an identifier's spelling, an enumeration's enumerators within its body, and an enumeration's items of C alone

In [ ]:
assert validate(lambda u: u.enumerator(lambda r: r)) == ["an enumerator needs a name"]
assert validate(enumerator("1A")) == ["'1A' is not an identifier"] and validate(enumerator("été")) == []
assert validate(type_name(lambda s: s.namedtypespecifier(lambda n: n.name(named("2t"))))) == ["'2t' is not an identifier"]
assert validate(type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").enumerators([enumerator("A")])))) == [
    "an EnumSpecifier with enumerators has a body"]
assert validate(type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").trailing_comma(True)))) == [
    "an EnumSpecifier with enumerators has a body"]
assert validate(type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").body(True)))) == []  # `enum {}`
# Its terms' own checks: a literal's, within an enumerator's value; a value's references are its scope's
assert validate(enumerator("A", C.literal(1, "q"))) == ["value: a literal's suffix cannot be 'q'"]
assert validate(enumerator("B", x)) == ["value: variable 'x' is not bound"] and validate(enumerator("B", x), {"x"}) == []
# An enumeration's items are its own dialect's terms: a directive's items, any dialect's by schema, are C Types' within C's
cpp_item = CppTypes.Expressions.DIALECT.spec(lambda u: u.enumerator(lambda r: r.name(spelled("B"))))
assert validate(enumeration([ifdef("B_", [cpp_item])])) == [
    "specifiers 0 argument 0: argument 0: a Cpp Types enumerator is not a C Types term"]
assert validate(enumeration([ifdef("B_", [enumerator("B")])])) == []


<a id="CTY-04"></a>
## CTY-04 · What it binds: nothing; the references an enumerator's value or a type name's terms hold are their scope's

In [ ]:
assert SY.free(term(enumeration([enumerator("A", C.literal(1)), enumerator("B", C.identifier("A"))])), CT.DIALECT) == {"A"}
sized = type_name(keyword("int"), declarator=lambda d: d.arraydeclarator(lambda a: a.size(C.binary("+", x, 1))))  # int[x + 1]
assert SY.free(term(sized), CT.DIALECT) == {"x"}
assert validate(sized) == ["declarator size: left: variable 'x' is not bound"] and validate(sized, {"x"}) == []
typeof = type_name(lambda s: s.typeofspecifier(lambda t: t.keyword("typeof").operand(x)))  # typeof(x)
assert SY.free(term(typeof), CT.DIALECT) == {"x"} and not any(F.is_binder(b.schema) for b in store.registered(CT.DIALECT.union).branches)


<a id="CTY-05"></a>
## CTY-05 · Within C's terms: a cast's, a compound literal's and a generic association's type, and sizeof's, a C Types term C reads, validates, infers, evaluates and prints; written by C's writers

In [ ]:
uint8 = type_name(lambda s: s.namedtypespecifier(lambda n: n.name(named("uint8_t"))))
cast = term(C.cast(uint8, C.binary("+", x, 1)), C.DIALECT)  # (uint8_t)(x + 1)
assert F.form(cast).kind == "castexpression" and [F.dialect_of(a) for a in F.form(cast).arguments] == [CT.DIALECT.union, C.DIALECT.union]
assert F.kind_of(F.form(cast).arguments[0])[0] == "typeid" and C.DIALECT.kind(F.form(cast).arguments[0])[2] is CT.DIALECT.kinds()["typeid"]
assert C.DIALECT.validate(cast, {"x"}) == [] and CText.ToText(cast) == "(uint8_t)(x + 1)" and CV.SIMPLIFIER.rules.evaluate(cast, CV.Scope({"x": 255})).value == 0
assert CText.ToText(term(CV.OfAny(cast, {"x": 255}).residual, C.DIALECT)) == "(uint8_t)256"  # a uint8_t has no literal: x folds
assert C.DIALECT.infer(cast, {"x": D.Anything}).name() == "uint8_t" and F.same(term(C.DIALECT.make(F.form(cast)), C.DIALECT), cast)
assert SY.free(cast, C.DIALECT) == {"x"}  # the type holds no reference: the operand's alone
for spec, source in [
    (lambda u: u.sizeofexpression(lambda r: r.operand(int_)), "sizeof(int)"),
    (lambda u: u.compoundliteralexpression(lambda r: r.type(int_).initializer(lambda i: i.items([C.literal(1)]))), "(int){1}"),
    (lambda u: u.genericselection(lambda r: r.controlling(x).associations([lambda a: a.type(int_).value(C.literal(1)),
                                                                              lambda a: a.value(C.literal(0))])), "_Generic(x, int: 1, default: 0)"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(enumeration([enumerator("A")]))), "sizeof(enum E {\n    A\n})"),
]:
    assert C.DIALECT.validate(term(spec, C.DIALECT), {"x"}) == [] and CText.ToText(term(spec, C.DIALECT)) == source, source
generic = F.form(term(lambda u: u.genericselection(lambda r: r.controlling(x).associations([lambda a: a.value(C.literal(0))])), C.DIALECT))
assert generic.arguments[1] is None and F.kind_of(generic.arguments[2])[0] == "literal"  # an association: a pair, its type absent
# From its own holder, its terms' C terms are read through C
chars = type_name(keyword("char"), declarator=lambda d: d.arraydeclarator(lambda a: a.size(C.literal(4))))  # char[4]
assert CT.DIALECT.infer(term(int_)) is D.Anything and CTText.ToText(term(chars)) == "char[4]" and validate(chars) == []


<a id="CTY-06"></a>
## CTY-06 · Invalid syntax: error and missing in its Term and among an enumeration's items, validated, printed as written, refused by name

In [ ]:
skipped = lambda u: u.error(lambda r: r.text("int int"))  # noqa: E731
expected = lambda u: u.missing(lambda r: r.kind("type"))  # noqa: E731
assert validate(skipped) == ["invalid syntax 'int int'"] and validate(expected) == ["missing type"]
assert text(skipped) == "int int" and text(expected) == ""
assert C.DIALECT.validate(term(lambda u: u.castexpression(lambda r: r.type(skipped).operand(C.literal(1))), C.DIALECT)) == [
    "type: invalid syntax 'int int'"]
with raises(NotEvaluated, match="C Types error is not simplified"):  # a cast reads its type: invalid syntax there refused
    CV.OfAny(term(lambda u: u.castexpression(lambda r: r.type(skipped).operand(C.literal(1))), C.DIALECT)).evaluate()
with raises(ValueError, match="C Types error has no text to print"):
    text(lambda u: u.error(lambda r: r))
broken = enumeration([enumerator("A"), lambda u: u.error(lambda r: r.text("B C"))])
assert validate(broken) == ["specifiers 0 argument 1: invalid syntax 'B C'"] and text(broken) == "enum E {\n    A\n    B C\n}"


<a id="CTY-07"></a>
## CTY-07 · Its editions: C's three, each by its designation and common name, each with both its kinds

In [ ]:
types = store.Instances.Expressions.C.Types
assert list(FS.standards(types)) == ["ISO/IEC 9899:2011", "ISO/IEC 9899:2018", "ISO/IEC 9899:2024"]
assert [FS.tags(s) for s in FS.standards(types).values()] == [
    ("ISO/IEC 9899:2011", "C11"), ("ISO/IEC 9899:2018", "C17"), ("ISO/IEC 9899:2024", "C23")]
assert FS.properties(FS.find(types, "C23")) == {"bodies": ("org.iso", "ch.iec"), "number": "9899", "revision": ("year", 2024),
                                               "names": ("C23",)}
assert CT.SINCE == {} and all([s.name for s in FS.kinds(edition)] == ["Expressions.C.Types.TypeId", "Expressions.C.Types.Enumerator"]
                              for edition in FS.standards(types).values())
own = {b.name: b.schema for b in store.registered(CT.DIALECT.union).branches}
assert all(tuple(FS.tags(s)[-1] for s in FS.of(types, own[tag]).values()) == (("C11", "C17", "C23") if tag in CT.TAGS else ())
           for tag in own)  # invalid syntax, and the preprocessor's kinds, in none of its editions
assert store.transaction(CTypes.define)[0]["C17"] is FS.find(types, "C17")  # run again, it writes nothing
assert Validators.Validate(store)(store.registered("Expressions.Framework.Standard"), FS.find(types, "C11")) == []


<a id="CTY-08"></a>
## CTY-08 · Printing: each type name as mbse-programs' printer writes it, an enumeration's body on lines of its own, its directives at their line's start

In [ ]:
pointer = lambda inner=None: lambda d: d.pointerdeclarator(lambda p: p if inner is None else p.declarator(inner))  # noqa: E731
for spec, source in [
    (int_, "int"), (type_name(keyword("unsigned"), keyword("long")), "unsigned long"),
    (type_name(lambda s: s.cvqualifier(lambda q: q.keyword("const")), keyword("char"), declarator=pointer()), "const char*"),
    (type_name(keyword("int"), declarator=lambda d: d.arraydeclarator(lambda a: a.declarator(pointer()).size(C.literal(4)))), "int(*)[4]"),
    (type_name(keyword("int"), declarator=lambda d: d.functiondeclarator(lambda f: f.declarator(lambda p: p.parenthesizeddeclarator(
        lambda q: q.declarator(pointer()))).parameters([lambda p: p.parameterdeclaration(lambda q: q.specifiers([keyword("int")])),
                                                     lambda p: p.ellipsisparameter(lambda e: e)]))), "int(*)(int, ...)"),
    (type_name(lambda s: s.classspecifier(lambda c: c.key("struct").name(named("S")).body(True))), "struct S {}"),
    (type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").name(named("E")))), "enum E"),
    (type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").body(True))), "enum {}"),
    (type_name(lambda s: s.enumspecifier(lambda e: e.key("enum").name(named("E")).base(type_name(keyword("char"))).body(True).enumerators([
        enumerator("A")]))), "enum E : char {\n    A\n}"),
    (enumeration([enumerator("A", C.literal(1)), enumerator("B")]), "enum E {\n    A = 1,\n    B\n}"),
    (enumeration([enumerator("A"), enumerator("B")], trailing=True), "enum E {\n    A,\n    B,\n}"),
    (enumeration([enumerator("A", C.binary("+", C.literal(1), C.literal(2))), lambda u: u.comment(lambda c: c.text(" last").trailing(True))]),
     "enum E {\n    A = 1 + 2 // last\n}"),
    (enumeration([lambda u: u.comment(lambda c: c.text(" first ").block(True)), enumerator("A"),
                  ifdef("WIDE", [enumerator("B"), enumerator("C")]), enumerator("D")]),
     "enum E {\n    /* first */\n    A,\n#ifdef WIDE\n    B,\n    C,\n#endif\n    D\n}"),
    (enumeration([enumerator("A"), ifdef("B_", [enumerator("B")])]), "enum E {\n    A\n#ifdef B_\n    B,\n#endif\n}"),
     # the last enumerator of its body's own without a comma, as mbse-programs writes it
    (enumeration([enumerator("A", lambda u: u.sizeofexpression(lambda r: r.operand(enumeration([enumerator("B")], word="F"))))]),
     "enum E {\n    A = sizeof(enum F {\n        B\n    })\n}"),  # an enumeration within, one level deeper
    (enumerator("A", C.literal(1)), "A = 1"), (lambda u: u.comment(lambda c: c.text(" note")), "// note"),
]:
    assert text(spec) == source, (source, text(spec))
assert CTText.ToText(term(C.cast(int_, x), C.DIALECT)) == CText.ToText(term(C.cast(int_, x), C.DIALECT)) == "(int)x"  # one printer


<a id="CTY-09"></a>
## CTY-09 · Evaluation: none (Q11); each of its kinds is refused by name by its family's evaluator, wherever it stands; a cast reads its type rather than evaluate it

In [ ]:
for spec, tag in [(int_, "typeid"), (enumerator("A"), "enumerator")]:  # C's evaluator, which reads C Types' terms
    with raises(NotEvaluated, match=f"C Types {tag} is not simplified"):
        CV.OfAny(term(spec), {}).evaluate()
with raises(NotEvaluated, match="C sizeofexpression is not simplified"):  # sizeof is refused, not its type
    CV.OfAny(term(lambda u: u.sizeofexpression(lambda r: r.operand(int_)), C.DIALECT)).evaluate()
assert CV.SIMPLIFIER.rules.evaluate(term(C.cast(type_name(keyword("unsigned"), keyword("char")), 300), C.DIALECT), CV.Scope()).value == 44  # the cast converts
with raises(NotEvaluated, match="C Types typeid is not simplified"):  # a type where an expression may stand, alone
    CV.OfAny(term(lambda u: u.typeid(lambda i: i.specifiers([keyword("int")])), C.DIALECT)).evaluate()
